In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/home/mariam/miniconda3/envs/paper_recommender/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/arxiv_cleaned.csv")

In [3]:
model =SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1003.18it/s]


In [4]:
embiddings = model.encode(
    df['text'].tolist(),
    show_progress_bar=True)

Batches: 100%|██████████| 1286/1286 [03:09<00:00,  6.80it/s]


In [5]:
embiddings.shape

(41127, 384)

In [6]:
similarity_scores = cosine_similarity(
    embiddings[0].reshape(1, -1),
    embiddings
).flatten()

In [7]:
similar_indices = similarity_scores.argsort()[::-1]

In [8]:
similar_indices = similar_indices[similar_indices != 0]

In [9]:
top_indices = similar_indices[:5]

In [10]:
df.iloc[top_indices][['titles', 'abstracts']]

,titles,abstracts
372,Graph Convolutional Networks with EigenPooling,"Graph neural networks, which generalize deep n..."
294,ASAP: Adaptive Structure Aware Pooling for Lea...,Graph Neural Networks (GNN) have been shown to...
21464,Accurate Learning of Graph Representations wit...,Graph neural networks have been widely used on...
21759,Graph Attention Networks with Positional Embed...,Graph Neural Networks (GNNs) are deep learning...
30894,Understanding Attention and Generalization in ...,We aim to better understand attention over nod...


In [11]:
recommendations = df.iloc[top_indices][['titles', 'abstracts']].copy()

recommendations['similarity_score'] = similarity_scores[top_indices]

recommendations

,titles,abstracts,similarity_score
372,Graph Convolutional Networks with EigenPooling,"Graph neural networks, which generalize deep n...",0.793818
294,ASAP: Adaptive Structure Aware Pooling for Lea...,Graph Neural Networks (GNN) have been shown to...,0.786501
21464,Accurate Learning of Graph Representations wit...,Graph neural networks have been widely used on...,0.785371
21759,Graph Attention Networks with Positional Embed...,Graph Neural Networks (GNNs) are deep learning...,0.781725
30894,Understanding Attention and Generalization in ...,We aim to better understand attention over nod...,0.776806


In [12]:
def recommend_papers_transformer(paper_index, top_k=5):
    
    # Calculate cosine similarity between the selected paper
    # and all papers in the dataset
    similarity_scores = cosine_similarity(
        embiddings[paper_index].reshape(1, -1),
        embiddings
    ).flatten()

    # Sort papers from highest similarity to lowest
    similar_indices = similarity_scores.argsort()[::-1]

    # Remove the input paper itself
    similar_indices = similar_indices[similar_indices != paper_index]

    # Select the top K similar papers
    top_indices = similar_indices[:top_k]

    # Create recommendations table
    recommendations = df.iloc[top_indices][
        ['titles', 'abstracts']
    ].copy()

    # Add similarity scores
    recommendations['similarity_score'] = similarity_scores[top_indices]

    return recommendations

In [13]:
recommend_papers_transformer(0, 5)

,titles,abstracts,similarity_score
372,Graph Convolutional Networks with EigenPooling,"Graph neural networks, which generalize deep n...",0.793818
294,ASAP: Adaptive Structure Aware Pooling for Lea...,Graph Neural Networks (GNN) have been shown to...,0.786501
21464,Accurate Learning of Graph Representations wit...,Graph neural networks have been widely used on...,0.785371
21759,Graph Attention Networks with Positional Embed...,Graph Neural Networks (GNNs) are deep learning...,0.781725
30894,Understanding Attention and Generalization in ...,We aim to better understand attention over nod...,0.776806


In [14]:
import numpy as np

In [15]:
np.save("../models/transformer_embeddings.npy", embiddings)